# 03 · Test, Error Analysis & Deploy 🔍

**Lab Phases 5–6 + Deploy / Monitor.**

- final evaluation on the **unseen test set** (overall + per class + confusion matrix)
- **error analysis**: at least 10 documented failure cases (false positive / false
  negative / wrong class / poor localization)
- a **demo** on new images
- **export** the model and hand off to the Streamlit app

In [ ]:
# %pip install -q ultralytics

In [ ]:
import os, sys, shutil
from pathlib import Path
cwd = os.getcwd()
ROOT = os.path.abspath(os.path.join(cwd, "..")) if os.path.basename(cwd).lower() == "notebooks" else cwd
sys.path.insert(0, os.path.join(ROOT, "src"))
import utils as U
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

MODELS_DIR = Path(ROOT) / "models"; RESULTS_DIR = Path(ROOT) / "results"
FAIL_DIR = RESULTS_DIR / "failures"; FAIL_DIR.mkdir(parents=True, exist_ok=True)

# ---- CONFIG ----
MODEL_PATH = MODELS_DIR / "best_v2.pt"
if not MODEL_PATH.exists():
    MODEL_PATH = MODELS_DIR / "best_v1.pt"
DATA_YAML = U.find_data_yaml(str(Path(ROOT) / "datasets"))
assert MODEL_PATH.exists(), "Train a model first (run 02_training.ipynb)."
assert DATA_YAML, "No data.yaml found under ./datasets."
print("Model:", MODEL_PATH)
print("Data :", DATA_YAML)

model = YOLO(str(MODEL_PATH))
data = U.load_data_yaml(DATA_YAML); names = data["names"]

## 1 · Final evaluation on the unseen test set

In [ ]:
metrics = model.val(data=DATA_YAML, split="test", conf=0.001, iou=0.6, plots=True,
                    project=os.path.join(ROOT, "runs"), name="eval_test", exist_ok=True)
overall, per_class = U.summarize_metrics(metrics)
print("TEST OVERALL:", {k: round(v, 3) for k, v in overall.items()})
per_class.to_csv(RESULTS_DIR / "test_metrics_per_class.csv", index=False)
per_class

## 2 · Confusion matrix & PR curve
Saved automatically by the evaluation above.

In [ ]:
save_dir = Path(metrics.save_dir)
for pic in ["confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png"]:
    p = save_dir / pic
    if p.exists():
        plt.figure(figsize=(6, 5)); plt.imshow(Image.open(p)); plt.axis("off"); plt.title(pic); plt.show()

## 3 · Error analysis — find where it breaks *(Lab Phase 6)*
Run the detector on every test image, match predictions to ground truth by IoU, and
bucket the mistakes: **false negative** (missed), **false positive** (hallucinated),
**wrong class**, **poor localization** (right class, loose box).

In [ ]:
CONF_EA = 0.25   # confidence used for qualitative error analysis
records = []
test_imgs = U.list_images(data["splits"]["test"])
for img_path in test_imgs:
    r = model.predict(img_path, conf=CONF_EA, iou=0.5, verbose=False)[0]
    H, W = r.orig_shape
    if r.boxes is not None and len(r.boxes):
        pb = r.boxes.xyxy.cpu().numpy()
        pc = r.boxes.cls.cpu().numpy().astype(int)
        pconf = r.boxes.conf.cpu().numpy()
    else:
        pb, pc, pconf = np.zeros((0, 4)), np.array([], int), np.array([])
    lab = U.read_yolo_label(U.label_path_for_image(img_path))
    gt_boxes = [U.xywhn_to_xyxy(row[1:5], W, H) for row in lab]
    gt_cls = lab[:, 0].astype(int).tolist()
    rec = U.match_detections(pb, pc, pconf, gt_boxes, gt_cls, iou_thr=0.5)
    for cat in ["false_negative", "false_positive", "wrong_class", "poor_localization"]:
        if rec[cat]:
            records.append({"image": img_path, "failure_type": cat, "count": len(rec[cat])})

fdf = pd.DataFrame(records)
if len(fdf):
    print("Test images with >=1 failure:", fdf["image"].nunique())
    display(fdf["failure_type"].value_counts())
else:
    print("No failures found — lower CONF_EA, or double-check the test labels.")

## 4 · Document ≥10 failure cases (Ground Truth vs Prediction)

In [ ]:
# pick a diverse set spanning all failure types, at least 10 images
chosen = []
if len(fdf):
    for cat in ["false_negative", "wrong_class", "poor_localization", "false_positive"]:
        for img in fdf[fdf.failure_type == cat]["image"].unique():
            if img not in chosen:
                chosen.append(img)
            if len(chosen) >= 12:
                break
        if len(chosen) >= 12:
            break
    for img in fdf["image"].unique():
        if len(chosen) >= 10:
            break
        if img not in chosen:
            chosen.append(img)

rows = []
for img_path in chosen:
    r = model.predict(img_path, conf=CONF_EA, iou=0.5, verbose=False)[0]
    H, W = r.orig_shape
    lab = U.read_yolo_label(U.label_path_for_image(img_path))
    gt_boxes = [U.xywhn_to_xyxy(row[1:5], W, H) for row in lab]
    gt_cls = lab[:, 0].astype(int).tolist()
    gt_img = U.draw_boxes(img_path, gt_boxes, [names[c] for c in gt_cls], class_ids=gt_cls)
    pred_img = Image.fromarray(r.plot()[:, :, ::-1])
    types = fdf[fdf.image == img_path]["failure_type"].unique().tolist()
    pred_names = [names[int(c)] for c in r.boxes.cls.cpu().numpy().astype(int)] if r.boxes is not None else []

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].imshow(gt_img); ax[0].set_title("Ground truth"); ax[0].axis("off")
    ax[1].imshow(pred_img); ax[1].set_title("Prediction — " + ", ".join(types)); ax[1].axis("off")
    out = FAIL_DIR / (Path(img_path).stem + "__" + "_".join(types) + ".png")
    plt.tight_layout(); plt.savefig(out, dpi=110); plt.show()

    rows.append({"image": Path(img_path).name,
                 "failure_type": ", ".join(types),
                 "predicted": ", ".join(pred_names),
                 "ground_truth": ", ".join(names[c] for c in gt_cls),
                 "what_went_wrong": "",
                 "why_it_happened": "",
                 "new_data_that_helps": ""})

fail_table = pd.DataFrame(rows)
fail_table.to_csv(RESULTS_DIR / "failure_cases.csv", index=False)
print("Saved", len(rows), "failure cases ->", FAIL_DIR, "and results/failure_cases.csv")
fail_table

## 5 · Turn failures into a plan *(Lab Phase 7)*
Open **`results/failure_cases.csv`** and fill in `what_went_wrong`, `why_it_happened`
and `new_data_that_helps` for each row. Then collect **30–50 targeted images**:

| Failure you saw | New data to collect |
|---|---|
| Small item missed | distant / crowded plate photos |
| Low-light failure | indoor / night images |
| Two classes confused | clearer, balanced examples of both |
| Occlusion failure | dishes partly covered by other items |

Add them in Roboflow as **version 2**, then retrain **v2** back in `02_training.ipynb`.

## 6 · Demo on unseen images

In [ ]:
DEMO_DIR = os.path.join(ROOT, "images-mixed")   # swap for any folder of NEW photos
demo_imgs = U.list_images(DEMO_DIR)[:6]
if demo_imgs:
    res = model.predict(demo_imgs, conf=0.35, verbose=False)
    cols = 3; nrows = (len(res) + cols - 1) // cols
    fig, axes = plt.subplots(nrows, cols, figsize=(5 * cols, 4 * nrows))
    axes = np.array(axes).ravel()
    for ax, r in zip(axes, res):
        ax.imshow(r.plot()[:, :, ::-1]); ax.axis("off")
        found = [names[int(c)] for c in r.boxes.cls] if r.boxes is not None else []
        ax.set_title(", ".join(found) if found else "nothing", fontsize=9)
    for ax in axes[len(res):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No demo images in", DEMO_DIR)

## 7 · Export the model for deployment

In [ ]:
# Copy the chosen weights to a stable name the Streamlit app loads by default:
shutil.copy(str(MODEL_PATH), str(MODELS_DIR / "model.pt"))
print("App will load:", MODELS_DIR / "model.pt")

# Optional — ONNX export (portable, fast on CPU). Uncomment to use:
# model.export(format="onnx")

## 8 · Deploy (Streamlit) & Monitor
Run the app from the **repo root**:

```bash
streamlit run app/app.py
```

Upload a food photo → get boxes + labels + confidence.

**Monitoring:** the app appends every prediction (time, file, class, confidence) to
`results/prediction_log.csv`. Periodically review the **low-confidence** predictions —
those are exactly the images to add to your next Roboflow round. That closes the loop:
*collect → train → find failures → collect better data → retrain.*